# Pertemuan 5 — Membersihkan Data, dengan Alasan
### Workbook belajar mandiri — versi dataset COVID-19 Indonesia

**Exploratory Data Analysis (KMTI21133)**  
**Sub-CPMK5:** Mahasiswa mampu menerapkan teknik *data preparation* dan *data manipulation* dengan Python
disertai justifikasi setiap keputusan (C3).

---

> **Dosen masih berhalangan hadir.** Kelas didampingi asisten. Workbook ini dapat Anda selesaikan sendiri.

> **Versi dataset ini:** seluruh sel mengikuti workbook P05 asli, tetapi datanya diganti dengan
> `covid_19_indonesia_time_series_all.csv` — kasus harian COVID-19 per provinsi di Indonesia
> (1 Maret 2020 – 16 September 2022). Angka pada kunci `cek(...)` sudah dihitung ulang untuk data ini.

| | |
|---|---|
| **Perkiraan waktu** | 150 menit. Bagian 1–7 sekitar 60 menit, Bagian 8 sekitar 30 menit, Bagian 9–13 sekitar 35 menit, Bagian 14 sekitar 25 menit. |
| **Prasyarat** | Workbook pertemuan 4 sudah selesai, termasuk Bagian 10 (profil kualitas data tim). |
| **Umpan balik** | Sama seperti pekan lalu: sel `cek(...)` menjawab BENAR atau BELUM tanpa menampilkan kuncinya. |

### Yang Anda hasilkan di akhir sesi

**M2 — Data Quality Report (bobot 10%)**, terdiri atas tiga berkas:

1. `reports/data_quality_report.docx` — laporan temuan dan keputusan
2. `notebooks/02_cleaning.ipynb` — notebook pembersihan yang dapat dijalankan ulang
3. `cleaning_log.md` — catatan setiap keputusan beserta alasannya

Pakai `cek_m2.py` untuk memeriksa kelengkapan kiriman sebelum mengumpulkan.

---
## Bagian 0 — Persiapan

In [1]:
import pandas as pd
import numpy as np
import hashlib, os, re, shutil

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 150)

FILE = 'covid_19_indonesia_time_series_all.csv'
RAW  = os.path.join('data', 'raw', FILE)          # data mentah: dibaca saja, tidak pernah diubah
os.makedirs('data/raw', exist_ok=True)

if not os.path.exists(RAW):
    if not os.path.exists(FILE):
        try:
            from google.colab import files
            unggahan = files.upload()               # pilih berkas CSV COVID-19 Anda
            FILE = list(unggahan.keys())[0]
        except ImportError:
            raise FileNotFoundError('Letakkan CSV di folder yang sama dengan notebook ini.')
    shutil.move(FILE, RAW)                          # simpan di data/raw/ sesuai aturan 1

with open(RAW, 'rb') as f:
    ada_bom = f.read(3) == b'\xef\xbb\xbf'

mentah = pd.read_csv(RAW, encoding='utf-8-sig')     # utf-8-sig membuang BOM di awal berkas
print('Data mentah:', mentah.shape)
print('BOM di awal berkas:', ada_bom, '(tanpa utf-8-sig, kolom pertama akan bernama "\\ufeffDate")')

Saving covid_19_indonesia_time_series_all.csv to covid_19_indonesia_time_series_all.csv
Data mentah: (31822, 38)
BOM di awal berkas: True (tanpa utf-8-sig, kolom pertama akan bernama "\ufeffDate")


In [2]:
_K = {
    'P5_pulau': '7902699be4',
    'P5_status': '4e07408562',
    'P5_lokasi': '9f14025af0',
    'P5_baris_bersih': '8e68b83c92',
    'P5_dibuang': 'e0e3fcdfb0',
    'P5_curiga': '9869a8a3a1',
    'P5_rate_kosong': '23c657f2ef',
    'P5_rate_num': '318b4b7811',
    'P5_bulan': 'eb1e33e8a8',
    'P5_langkah': '35135aaa6c'
}

_P = {
    'P5_pulau': 'nunique() kolom island setelah diseragamkan (nilai kosong tidak ikut dihitung).',
    'P5_status': 'nunique() kolom special_status; nilai kosong tidak ikut dihitung.',
    'P5_lokasi': 'nunique() kolom location setelah strip. Baris nasional "Indonesia" masih ikut pada tahap ini.',
    'P5_baris_bersih': 'len() setelah seluruh langkah pembuangan/pemisahan baris dijalankan.',
    'P5_dibuang': 'Jumlah baris mentah dikurangi jumlah baris bersih.',
    'P5_curiga': 'Hitung baris yang bertanda kumulatif_janggal == True.',
    'P5_rate_kosong': 'Setelah rasio > 100% diubah menjadi NaN, hitung isna() pada case_recovered_rate data bersih.',
    'P5_rate_num': 'Berapa baris yang case_fatality_rate-nya berupa angka (notna) pada data bersih.',
    'P5_bulan': 'nunique() kolom bulan hasil dt.to_period(\'M\').',
    'P5_langkah': 'Berapa baris yang ada di cleaning log Anda.'
}

def _h(x):
    return hashlib.sha256(str(x).strip().lower().encode()).hexdigest()[:10]

def cek(tugas, jawaban):
    if tugas not in _K:
        print('Nama tugas tidak dikenal:', tugas); return
    if _h(jawaban) == _K[tugas]:
        print(f'BENAR   {tugas} = {jawaban}')
    else:
        print(f'BELUM   {tugas} = {jawaban}')
        print('        Petunjuk:', _P.get(tugas, '-'))

print('Alat pemeriksa siap.')

Alat pemeriksa siap.


---
## Bagian 1 — Pembersihan adalah keputusan analitis

Pekan lalu Anda menemukan belasan masalah. Hari ini setiap masalah itu menuntut sebuah **keputusan**,
dan hampir tidak ada yang punya jawaban tunggal.

Ambil satu contoh: 288 baris provinsi memiliki `total_recovered` **lebih besar** daripada `total_cases` —
jumlah orang sembuh melebihi jumlah orang yang pernah tercatat positif. Ada empat pilihan, semuanya masuk akal:

| Pilihan | Masuk akal bila | Risikonya |
|---|---|---|
| Dibuang barisnya | Yakin itu kesalahan input | Kehilangan hari pelaporan yang mungkin sah, padahal kolom lain di baris itu benar |
| Dijadikan kosong | Belum tahu penyebabnya | Analisis berkurang datanya |
| Dikoreksi (misalnya sembuh dipaksa sama dengan kasus) | Yakin itu keterlambatan pelaporan yang bisa dirapikan | Mengarang data bila dugaan salah |
| Dibiarkan, tetapi ditandai | Ingin menyerahkan keputusan ke tahap analisis | Harus diingat terus |

> **Yang dinilai bukan pilihan Anda, melainkan apakah Anda punya alasan dan menuliskannya.**
> Dua tim boleh mengambil keputusan berbeda pada data yang sama dan keduanya memperoleh nilai penuh,
> asalkan alasannya jelas. Tim yang membersihkan tanpa alasan tertulis tidak memperoleh nilai penuh
> meskipun kodenya rapi.

### Tiga aturan yang berlaku sepanjang pertemuan ini

1. **Jangan pernah menimpa data mentah.** `data/raw/` tidak pernah diubah. Hasil disimpan di `data/processed/`.
2. **Satu langkah, satu catatan.** Setiap perubahan masuk ke *cleaning log* saat itu juga, bukan di akhir.
3. **Menandai lebih aman daripada menebak.** Kalau Anda tidak tahu nilai yang benar, jangan mengarang.
   Jadikan kosong atau beri kolom penanda.

---
## Bagian 2 — Urutan pembersihan

Urutan bukan selera. Langkah yang dikerjakan terlalu awal akan membuat langkah berikutnya salah.

| Urutan | Langkah | Mengapa harus di sini |
|---|---|---|
| 1 | Perbaiki **tipe data** | Selama tanggal masih teks (`3/1/2020`) dan persentase masih `'51.28%'`, tidak satu pun pemeriksaan angka bisa jalan |
| 2 | Buang **spasi** pada teks | Kalau tidak, penyeragaman kategori akan meleset |
| 3 | Ubah **placeholder** menjadi kosong | Kalau tidak, kode seperti `-999` ikut terhitung sebagai nilai |
| 4 | **Seragamkan** kategori | Setelah spasi bersih, barulah sinonim terlihat |
| 5 | Tangani **duplikat** (termasuk duplikat tingkat agregasi) | Setelah teks seragam, duplikat nyaris baru bisa dikenali; baris nasional harus dipisah sebelum ada penjumlahan |
| 6 | Tangani **nilai mustahil** | Perlu tipe data yang benar untuk membandingkan |
| 7 | Tangani **nilai kosong** | Paling akhir, karena langkah 3 dan 6 menambah nilai kosong baru |
| 8 | Buat **kolom turunan** | Hanya bermakna di atas data yang sudah bersih dan terurut |

> **Kesalahan urutan yang paling sering terjadi:** mengisi nilai kosong lebih dulu, lalu baru mengubah
> placeholder menjadi kosong. Pada data deret waktu ada versi lain: menghitung rata-rata 7 hari sebelum
> baris nasional dipisahkan dan data diurutkan, sehingga angka Indonesia tercampur dengan angka provinsi.

---
## Bagian 3 — Langkah 1 dan 2: tipe data

In [3]:
# CONTOH 3.1 — salin dulu, jangan sentuh yang mentah
d = mentah.copy()

jejak = []
def catat(langkah, terdampak, alasan):
    """Setiap perubahan dicatat SAAT dikerjakan, bukan belakangan."""
    jejak.append({'langkah': langkah, 'baris_terdampak': int(terdampak), 'alasan': alasan})
    print(f'  {langkah}: {int(terdampak)}')

def snake(nama):
    return re.sub(r'[^a-z0-9]+', '_', nama.lower()).strip('_')

d.columns = [snake(c) for c in d.columns]
catat('Seragamkan nama kolom ke snake_case', len(d.columns),
      'Nama berspasi, berkurung, dan berkapital (mis. "Area (km2)") menyulitkan kode; yang dihitung di sini KOLOM, bukan baris')
print('Contoh:', list(mentah.columns[[0, 1, 25, 33]]), '->', list(d.columns[[0, 1, 25, 33]]))
print('Siap. Data mentah tetap utuh di variabel `mentah`.')

  Seragamkan nama kolom ke snake_case: 38
Contoh: ['Date', 'Location ISO Code', 'Population', 'Total Deaths per 100rb'] -> ['date', 'location_iso_code', 'population', 'total_deaths_per_100rb']
Siap. Data mentah tetap utuh di variabel `mentah`.


In [4]:
# CONTOH 3.2 — persentase: dari teks ke angka
for kol in ['case_fatality_rate', 'case_recovered_rate']:
    berformat = d[kol].astype(str).str.endswith('%')

    d[kol] = pd.to_numeric(
        d[kol].astype(str)
          .str.replace('%', '', regex=False)     # titik di sini desimal (51.28), BEDA dengan titik ribuan pada Rupiah
          .str.strip(),
        errors='coerce')

    catat(f'Konversi {kol} ke angka', berformat.sum(),
          'Buang "%" agar kolom dapat dihitung; satuan tetap persen (0-100), bukan pecahan')
    print('    Gagal dikonversi:', int(d[kol].isna().sum()))

  Konversi case_fatality_rate ke angka: 31822
    Gagal dikonversi: 0
  Konversi case_recovered_rate ke angka: 31822
    Gagal dikonversi: 0


> **Perhatikan `errors='coerce'`.** Nilai yang tidak bisa dikonversi menjadi `NaN` alih-alih menghentikan
> program. Itu sengaja: kita ingin tahu berapa banyak yang gagal, bukan berhenti di nilai pertama yang aneh.
> Selalu cetak jumlah kegagalannya. Kalau angkanya besar, berarti asumsi format Anda salah.
>
> **Jangan menyalin langkah lama mentah-mentah.** Pada workbook koperasi, `.replace('.', '')` benar karena
> titik adalah pemisah ribuan. Di sini titik adalah **desimal**; menghapusnya akan mengubah `51.28%`
> menjadi `5128`.

In [5]:
# CONTOH 3.3 — tanggal: format bulan/hari/tahun, dan bukti bahwa kita tidak menebak
t = d['date'].astype(str)

bagian = t.str.split('/', expand=True).astype(int)
print('Nilai maksimum bagian-1:', bagian[0].max(), '| bagian-2:', bagian[1].max(), '| bagian-3:', bagian[2].max())
# bagian-1 tidak pernah melebihi 12 dan bagian-2 mencapai 31 -> urutannya bulan/hari/tahun, bukan hari/bulan/tahun

d['date'] = pd.to_datetime(t, format='%m/%d/%Y', errors='coerce')

catat('Konversi date dengan format %m/%d/%Y', len(d),
      'dayfirst=True akan menukar bulan dan hari (3/1/2020 menjadi 3 Januari) tanpa memunculkan galat')

print('Gagal   :', int(d['date'].isna().sum()))
print('Rentang :', d['date'].min().date(), 'sampai', d['date'].max().date())
print('Hari unik:', d['date'].nunique(), '| bulan unik:', d['date'].dt.to_period('M').nunique())

Nilai maksimum bagian-1: 12 | bagian-2: 31 | bagian-3: 2022
  Konversi date dengan format %m/%d/%Y: 31822
Gagal   : 0
Rentang : 2020-03-01 sampai 2022-09-16
Hari unik: 930 | bulan unik: 31


> Uji akal sehat dari pekan lalu dijalankan lagi di sini: wabah tercatat mulai Maret 2020 dan data berhenti
> September 2022, jadi 31 bulan berurutan. Kalau Anda salah membaca urutan tanggal, rentang ini akan
> melompat-lompat. Benar.

---
## Bagian 4 — Langkah 2 sampai 4: teks, placeholder, kategori

In [6]:
# CONTOH 4.1 — buang spasi pada SELURUH kolom teks sekaligus
kolom_teks = [c for c in d.columns
              if d[c].dtype == object or pd.api.types.is_string_dtype(d[c])]

n_spasi = 0
for kol in kolom_teks:
    s = d[kol].astype('string')
    n_spasi += int((s != s.str.strip()).sum())
    d[kol] = s.str.strip()

catat('Buang spasi awal dan akhir kolom teks', n_spasi,
      'Spasi tersembunyi memecah groupby dan melambungkan nunique')
print('Kolom teks yang diperiksa:', len(kolom_teks))

  Buang spasi awal dan akhir kolom teks: 0
Kolom teks yang diperiksa: 9


In [7]:
# CONTOH 4.2 — placeholder menjadi kosong
CURIGA = ['n/a', 'na', '-', '', 'null', 'none', 'tidak diketahui', 'unknown', '?', '#n/a']

n_ph = 0
for kol in kolom_teks:
    s = d[kol].astype('string')
    m = s.str.lower().isin(CURIGA).fillna(False)
    n_ph += int(m.sum())
    if m.any():
        d.loc[m, kol] = pd.NA
catat('Placeholder teks menjadi kosong', n_ph, 'N/A, -, null, dan teks kosong bukan nilai sungguhan')

# Kode sentinel numerik. Hanya kolom hitungan yang tak boleh negatif yang diperiksa:
# new_active_cases dan total_active_cases SAH bernilai negatif, jadi -999 di sana tidak bisa langsung dianggap placeholder.
HITUNGAN = ['new_cases', 'new_deaths', 'new_recovered', 'total_cases', 'total_deaths', 'total_recovered']
SENTINEL = [-999, -9999]
n_sentinel = 0
for kol in HITUNGAN:
    m = d[kol].isin(SENTINEL)
    n_sentinel += int(m.sum())
    if m.any():
        d.loc[m, kol] = np.nan
catat('Kode sentinel -999/-9999 pada kolom hitungan menjadi kosong', n_sentinel,
      'Kode semacam ini merusak rata-rata tanpa pernah terdeteksi isna(); diperiksa dan tidak ditemukan')

  Placeholder teks menjadi kosong: 0
  Kode sentinel -999/-9999 pada kolom hitungan menjadi kosong: 0


In [8]:
# CONTOH 4.3 — penyeragaman kategori, dengan peta keputusan yang eksplisit
KOLOM_KATEGORI = ['location_level', 'location', 'province', 'island', 'time_zone', 'special_status', 'location_iso_code']

# Bagian yang bisa dikerjakan mesin: HANYA mendeteksi varian huruf/spasi. Huruf TIDAK diubah otomatis,
# karena .str.title() akan merusak 'DKI Jakarta' -> 'Dki Jakarta' dan 'UTC+07:00' -> 'Utc+07:00'.
print(f'{"kolom":16s} {"unik":>5s} {"unik setelah casefold+rapikan spasi":>38s}')
for kol in KOLOM_KATEGORI:
    s = d[kol].dropna().astype(str)
    varian = s.str.casefold().str.replace(r'\s+', ' ', regex=True).nunique()
    print(f'{kol:16s} {s.nunique():5d} {varian:38d}')

# Bagian yang HARUS diputuskan manusia, ditulis sebagai peta agar bisa ditinjau orang lain.
# Hasil pemeriksaan: tidak ada sinonim atau salah ketik, jadi peta dibiarkan kosong. Contoh bentuknya bila ada:
#   {'DI Yogyakarta': 'Daerah Istimewa Yogyakarta'}
PETA_KATEGORI = {}
sebelum = d['location'].nunique()
d['location'] = d['location'].replace(PETA_KATEGORI)

catat('Seragamkan kategori', sebelum - d['location'].nunique(),
      'Tidak ada sinonim/salah ketik pada nama lokasi; peta kosong adalah temuan, bukan kelalaian. Huruf tidak diubah otomatis')

# Konsistensi: satu lokasi harus punya satu kode ISO, satu pulau, satu zona waktu
print()
print('Maksimum nilai berbeda per lokasi (harus 1):')
print(d.groupby('location')[['location_iso_code', 'island', 'time_zone', 'special_status']].nunique(dropna=False).max())

kolom             unik    unik setelah casefold+rapikan spasi
location_level       2                                      2
location            35                                     35
province            34                                     34
island               7                                      7
time_zone            3                                      3
special_status       3                                      3
location_iso_code    35                                     35
  Seragamkan kategori: 0

Maksimum nilai berbeda per lokasi (harus 1):
location_iso_code    1
island               1
time_zone            1
special_status       1
dtype: int64


> **Mengapa peta ditulis sebagai dictionary, bukan rangkaian `replace` berantai?** Karena peta bisa dibaca
> orang lain, bisa ditinjau, bisa ditambah, dan ikut masuk ke laporan sebagai lampiran. Rangkaian `replace`
> yang tersebar di beberapa sel tidak bisa.
>
> Di dataset ini peta itu kosong, dan itu **hasil pemeriksaan**, bukan alasan untuk melewatkannya. Perhatikan
> juga bahwa kita sengaja **tidak** memakai `.str.title()` seperti pada workbook koperasi: langkah yang aman di
> satu dataset bisa merusak dataset lain. Setiap langkah harus diuji pada data yang sedang Anda pegang.

In [9]:
# CONTOH 4.4 — kolom teks lain: lihat nilai uniknya, jangan percaya begitu saja
for kol in ['location_level', 'island', 'time_zone', 'special_status']:
    print(f'{kol:15s}:', sorted(d[kol].dropna().unique()))

location_level : ['Country', 'Province']
island         : ['Jawa', 'Kalimantan', 'Maluku', 'Nusa Tenggara', 'Papua', 'Sulawesi', 'Sumatera']
time_zone      : ['UTC+07:00', 'UTC+08:00', 'UTC+09:00']
special_status : ['Daerah Istimewa', 'Daerah Khusus', 'Daerah Khusus Ibu Kota']


### TUGAS 1 — hasil penyeragaman

In [10]:
# TUGAS 1
p5_pulau  = d['island'].nunique()                    # Jumlah pulau unik (nilai kosong tidak dihitung)
p5_status = d['special_status'].nunique()            # Jumlah status khusus unik (nilai kosong tidak dihitung)
p5_lokasi = d['location'].str.strip().nunique()      # Jumlah lokasi unik; baris nasional 'Indonesia' masih ikut

cek('P5_pulau', p5_pulau)
cek('P5_status', p5_status)
cek('P5_lokasi', p5_lokasi)

BENAR   P5_pulau = 7
BENAR   P5_status = 3
BENAR   P5_lokasi = 35


---
## Bagian 5 — Langkah 5: duplikat

In [11]:
# CONTOH 5.1 — duplikat persis
n_persis = int(d.duplicated().sum())
d = d.drop_duplicates()
catat('Hapus duplikat persis', n_persis, 'Seluruh kolom identik; satu baris disimpan (hasil pemeriksaan: tidak ada)')

  Hapus duplikat persis: 0


> Pada workbook koperasi, jumlah duplikat persis **membesar** setelah teks diseragamkan. Di sini jawabannya
> nol, dan itu juga informasi: data ini berasal dari sumber yang sudah terstruktur, sehingga tidak ada
> baris tersalin dua kali. Memeriksa lalu menemukan nol tetap dicatat, karena itu bukti bahwa pemeriksaan
> pernah dilakukan.

In [12]:
# CONTOH 5.2 — duplikat yang tersisa pada kunci utama
# Kunci utama data deret waktu per lokasi = (tanggal, lokasi): satu lokasi hanya boleh punya satu baris per hari.
n_id = int(d.duplicated(subset=['date', 'location'], keep=False).sum())
print('Baris dengan kunci (date, location) berulang:', n_id)

# Aturan penyatuan: simpan baris dengan total_cases terbesar, karena angka kumulatif tidak pernah turun
# sehingga pelaporan yang lebih besar adalah yang paling mutakhir.
d = (d.sort_values('total_cases', ascending=False)
       .drop_duplicates(subset=['date', 'location'], keep='first')
       .sort_index())
catat('Satukan duplikat menurut (date, location)', n_id,
      'Satu lokasi hanya boleh satu baris per hari; dipilih baris dengan total_cases terbesar (hasil pemeriksaan: tidak ada)')

Baris dengan kunci (date, location) berulang: 0
  Satukan duplikat menurut (date, location): 0


In [13]:
# CONTOH 5.3 — kunci utama yang kosong
n_kosong = int(d[['date', 'location']].isna().any(axis=1).sum())
d = d[d[['date', 'location']].notna().all(axis=1)]
catat('Buang baris tanpa date atau location', n_kosong,
      'Kunci utama kosong tidak dapat ditelusuri kembali ke sumber (hasil pemeriksaan: tidak ada)')

  Buang baris tanpa date atau location: 0


In [14]:
# CONTOH 5.4 — tanggal bolong: apakah ada hari yang hilang di tengah deret?
rentang = d.groupby('location')['date'].agg(['min', 'max', 'count'])
harapan = (rentang['max'] - rentang['min']).dt.days + 1
n_bolong = int((harapan - rentang['count']).sum())

catat('Periksa hari yang hilang di tengah deret tiap lokasi', n_bolong,
      'Tidak ada hari bolong. Lokasi baru muncul saat kasus pertamanya, itu BUKAN data hilang melainkan belum ada, jadi tidak diisi nol')
print('Lokasi mulai paling awal :', rentang['min'].min().date(), rentang['min'].idxmin())
print('Lokasi mulai paling akhir:', rentang['min'].max().date(), rentang['min'].idxmax())

  Periksa hari yang hilang di tengah deret tiap lokasi: 0
Lokasi mulai paling awal : 2020-03-01 DKI Jakarta
Lokasi mulai paling akhir: 2020-04-10 Gorontalo


In [15]:
# CONTOH 5.5 — duplikat tingkat agregasi: baris nasional 'Indonesia' adalah JUMLAH dari provinsi
print(d['location_level'].value_counts())
print()

nasional = d[d['location_level'] == 'Country'].copy()
n_nas = len(nasional)
d = d[d['location_level'] == 'Province'].copy()

catat('Pisahkan baris tingkat nasional ke tabel sendiri', n_nas,
      'Baris Country adalah agregat provinsi; bila dijumlahkan bersama baris provinsi, setiap kasus terhitung dua kali. Disimpan terpisah, tidak dihapus')

print('Total new_cases semua provinsi :', f"{d['new_cases'].sum():,}")
print('Total new_cases baris nasional :', f"{nasional['new_cases'].sum():,}")
print('Selisih (kasus tak terpetakan ke provinsi):', f"{nasional['new_cases'].sum() - d['new_cases'].sum():,}")

location_level
Province    30893
Country       929
Name: count, dtype: Int64

  Pisahkan baris tingkat nasional ke tabel sendiri: 929
Total new_cases semua provinsi : 6,397,309
Total new_cases baris nasional : 6,405,044
Selisih (kasus tak terpetakan ke provinsi): 7,735


> **Ini duplikat jenis lain, dan lebih berbahaya.** Tidak ada dua baris yang identik, tetapi satu baris
> (`Indonesia`) merupakan jumlah dari 34 baris lain. Rata-rata, total, atau grafik yang mencampur keduanya akan
> menggandakan hitungan tanpa memunculkan galat apa pun.
>
> Keputusannya: baris nasional **dipisah, bukan dibuang**. Ia berguna sebagai pembanding, misalnya untuk
> memeriksa bahwa jumlah provinsi mendekati angka nasional. Selisihnya menunjukkan sebagian kasus tidak
> dipetakan ke provinsi mana pun, dan itu layak dilaporkan sebagai keterbatasan.

---
## Bagian 6 — Langkah 6: nilai mustahil

Pohon keputusan yang dipakai di mata kuliah ini:

```
Nilai ini mustahil.
│
├─ Tahu nilai yang benar?            -> perbaiki, lalu catat sumber kebenarannya
├─ Tahu pasti ini kesalahan input?   -> jadikan kosong
├─ Mungkin sah tetapi belum jelas?   -> biarkan, tambahkan kolom penanda
└─ Barisnya tidak bisa dipakai       -> buang, lalu catat berapa banyak
```

Yang **tidak** ada di pohon ini: menebak. Kalau Anda tidak tahu, jangan mengarang nilai.

In [16]:
# CONTOH 6.1 — rasio di atas 100%: dijadikan kosong, bukan dibuang
# Rasio sembuh atau rasio kematian lebih dari 100% mustahil secara matematis.
for kol in ['case_recovered_rate', 'case_fatality_rate']:
    buruk = d[kol] > 100
    n_r = int(buruk.sum())
    d.loc[buruk, kol] = np.nan
    catat(f'{kol} > 100% menjadi kosong', n_r,
          'Rasio mustahil; berasal dari total sembuh/meninggal yang melebihi total kasus. Barisnya masih berguna untuk kolom lain')

  case_recovered_rate > 100% menjadi kosong: 288
  case_fatality_rate > 100% menjadi kosong: 43


In [17]:
# CONTOH 6.2 — hitungan negatif pada kolom yang tidak boleh negatif
# Catatan: new_active_cases BOLEH negatif (aktif turun karena banyak yang sembuh), jadi TIDAK diperiksa di sini.
print('new_active_cases < 0 (SAH, dibiarkan):', int((d['new_active_cases'] < 0).sum()), 'baris')

n_neg = 0
for kol in HITUNGAN:
    buruk = d[kol] < 0
    n_neg += int(buruk.sum())
    if buruk.any():
        d.loc[buruk, kol] = np.nan
catat('Hitungan harian/kumulatif negatif menjadi kosong', n_neg,
      'Jumlah orang tidak bisa negatif; hasil pemeriksaan: tidak ada. Selisih negatif pada new_active_cases sengaja tidak disentuh')

n_demo = int(((d['population'] <= 0) | (d['area_km2'] <= 0)).sum())
catat('Periksa populasi dan luas wilayah <= 0', n_demo, 'Penyebut untuk kolom per juta penduduk; hasil pemeriksaan: tidak ada')

new_active_cases < 0 (SAH, dibiarkan): 12696 baris
  Hitungan harian/kumulatif negatif menjadi kosong: 0
  Periksa populasi dan luas wilayah <= 0: 0


In [18]:
# CONTOH 6.3 — pelanggaran aturan: DITANDAI, bukan ditimpa
# Identitas akuntansi: aktif = kasus - meninggal - sembuh. Apakah kolom total_active_cases memenuhinya?
aktif_seharusnya = d['total_cases'] - d['total_deaths'] - d['total_recovered']
n_selisih = int((d['total_active_cases'] - aktif_seharusnya).abs().gt(0).sum())
catat('Uji identitas total_active = cases - deaths - recovered', n_selisih,
      'Identitas selalu terpenuhi, jadi aktif negatif adalah AKIBAT sembuh > kasus, bukan kesalahan hitung terpisah')

d['kumulatif_janggal'] = ((d['total_recovered'] > d['total_cases']) |
                          (d['total_deaths']    > d['total_cases']) |
                          (d['total_active_cases'] < 0))

catat('Tandai baris kumulatif janggal', d['kumulatif_janggal'].sum(),
      'Sembuh/meninggal melebihi total kasus atau aktif negatif; ditandai, tidak ditimpa, karena belum diketahui kolom mana yang keliru')

print(d.loc[d['kumulatif_janggal'], 'location'].value_counts().head(5))
d.loc[d['kumulatif_janggal'], ['date', 'location', 'total_cases', 'total_deaths', 'total_recovered', 'total_active_cases']].head(5)

  Uji identitas total_active = cases - deaths - recovered: 0
  Tandai baris kumulatif janggal: 479
location
Jawa Timur           132
Banten               103
DKI Jakarta           26
Jawa Barat            25
Kalimantan Tengah     25
Name: count, dtype: Int64


,date,location,total_cases,total_deaths,total_recovered,total_active_cases
0,2020-03-01,DKI Jakarta,39,20,75,-56
1,2020-03-02,DKI Jakarta,41,20,75,-54
4,2020-03-03,DKI Jakarta,43,20,75,-52
6,2020-03-03,Jawa Barat,1,1,60,-60
8,2020-03-04,DKI Jakarta,45,20,75,-50


> **Mengapa ditandai, bukan diperbaiki?** Karena kita tidak tahu kolom mana yang salah. Bisa `total_recovered`
> yang terlalu besar (data sembuh lama baru dilaporkan serentak), bisa `total_cases` yang terlambat dilaporkan.
> Memaksa `total_recovered = total_cases` sama saja dengan memutuskan bahwa angka kasus pasti benar, tanpa dasar apa pun.
>
> Kolom penanda memindahkan keputusan ke tahap analisis, di mana dampaknya bisa diuji:
> jalankan analisis dengan dan tanpa baris bertanda, lalu lihat apakah kesimpulannya berubah.

In [19]:
# CONTOH 6.4 — konsistensi harian vs kumulatif
# Total tidak boleh turun, dan selisih total antar hari harus sama dengan kolom new_*.
urut = d.sort_values(['location', 'date'])
n_tidak_konsisten = 0
for tot, baru in [('total_cases', 'new_cases'), ('total_deaths', 'new_deaths'), ('total_recovered', 'new_recovered')]:
    selisih = urut.groupby('location')[tot].diff().reindex(d.index)
    n_tidak_konsisten += int((selisih < 0).sum())
    n_tidak_konsisten += int((selisih.notna() & (selisih != d[baru])).sum())

catat('Uji konsistensi kolom kumulatif dan harian', n_tidak_konsisten,
      'Total tidak pernah turun dan selisihnya sama dengan new_*; hasil pemeriksaan: konsisten. Anomali ada di rasio sembuh/kasus, bukan di hitungan harian')

  Uji konsistensi kolom kumulatif dan harian: 0


---
## Bagian 7 — Langkah 7: nilai kosong, dan mengapa tidak ada pilihan yang netral

Empat strategi yang lazim dipakai:

| Strategi | Caranya | Cocok untuk |
|---|---|---|
| Buang barisnya | `dropna()` | MCAR, dan hanya bila yang hilang sedikit |
| Isi nilai tengah | `fillna(mean)` atau `fillna(median)` | MCAR, kolom yang bukan fokus analisis |
| Isi per kelompok | `groupby().transform()` | MAR, bila kelompoknya memang penjelasnya |
| Tandai sebagai kategori | `fillna('Tidak tercatat')` | Kolom kategorikal; kekosongan itu sendiri bermakna |

Dataset COVID ini **hampir tidak punya nilai kosong yang tersembunyi**: kekosongannya bersifat struktural
(wilayah memang tidak punya kota atau desa) atau akibat pembagian dengan nol. Karena itu pertanyaan
"bagaimana kalau ada nilai kosong yang acak atau yang disembunyikan?" kita uji lewat **simulasi**.

### Tidak ada kunci jawaban pada data nyata — jadi kita buat sendiri

Pada workbook koperasi tersedia berkas `P05_nilai_sebenarnya.csv`. Pada proyek Anda berkas semacam itu
tidak akan pernah ada. Triknya: ambil kolom yang **lengkap** (`new_cases`), sembunyikan sebagian nilainya
dengan dua mekanisme, lalu ukur seberapa jauh setiap strategi meleset dari nilai yang sebenarnya kita pegang.

- **MCAR** — 15% nilai dihapus **acak**, tidak peduli besar kecilnya.
- **MNAR** — nilai **besar** lebih sering dihapus (peluang 28% untuk nilai di atas median, 4% di bawahnya),
  mirip anggota bersaldo besar yang enggan menampilkan saldonya.

In [20]:
# CONTOH 7.1 — eksperimen imputasi (simulasi pada data nyata)
asli = d['new_cases'].astype(float).reset_index(drop=True)

rng = np.random.default_rng(42)
mcar = pd.Series(rng.random(len(asli)) < 0.15)
mnar = pd.Series(rng.random(len(asli)) < np.where(asli > asli.median(), 0.28, 0.04))

def ringkas(seri):
    return seri.describe()[['mean', '50%', 'std', 'max']]

def tabel_hasil(mask):
    obs = asli.where(~mask)
    hasil = pd.DataFrame({
        'KEBENARAN (semua baris)': ringkas(asli),
        'Hanya yang terlihat'    : ringkas(obs.dropna()),
        'Diisi dengan mean'      : ringkas(obs.fillna(obs.mean())),
        'Diisi dengan median'    : ringkas(obs.fillna(obs.median())),
    }).T.round(0)
    hasil.columns = ['mean', 'median', 'std', 'max']
    return hasil

print(f'MCAR: {mcar.mean():.1%} nilai disembunyikan secara acak')
display(tabel_hasil(mcar))
print()
print(f'MNAR: {mnar.mean():.1%} nilai disembunyikan, yang besar lebih sering')
display(tabel_hasil(mnar))

MCAR: 14.8% nilai disembunyikan secara acak


,mean,median,std,max
KEBENARAN (semua baris),207.0,24.0,772.0,16251.0
Hanya yang terlihat,207.0,24.0,770.0,15196.0
Diisi dengan mean,207.0,42.0,711.0,15196.0
Diisi dengan median,180.0,24.0,714.0,15196.0



MNAR: 16.0% nilai disembunyikan, yang besar lebih sering


,mean,median,std,max
KEBENARAN (semua baris),207.0,24.0,772.0,16251.0
Hanya yang terlihat,176.0,16.0,706.0,16251.0
Diisi dengan mean,176.0,29.0,647.0,16251.0
Diisi dengan median,150.0,16.0,650.0,16251.0


In [21]:
# CONTOH 7.2 — nilai yang disembunyikan itu seperti apa, dan seberapa jauh tiap strategi meleset?
for nama, m in [('MCAR (acak)', mcar), ('MNAR (yang besar disembunyikan)', mnar)]:
    obs = asli.where(~m)
    print(f'=== {nama} ===')
    print('  new_cases yang TERLIHAT     — rata-rata:', f'{obs.dropna().mean():>8,.1f}')
    print('  new_cases yang DISEMBUNYIKAN— rata-rata:', f'{asli[m].mean():>8,.1f}')
    print('  Rasio:', round(asli[m].mean() / obs.dropna().mean(), 2), 'kali lipat')
    for strategi, seri in [('Buang barisnya', obs.dropna()),
                           ('Isi dengan mean', obs.fillna(obs.mean())),
                           ('Isi dengan median', obs.fillna(obs.median()))]:
        salah = abs(seri.mean() - asli.mean()) / asli.mean() * 100
        print(f'  {strategi:18s} meleset {salah:5.1f}% dari rata-rata sebenarnya | std {seri.std():7.1f} (asli {asli.std():.1f})')
    print()

=== MCAR (acak) ===
  new_cases yang TERLIHAT     — rata-rata:    206.7
  new_cases yang DISEMBUNYIKAN— rata-rata:    209.3
  Rasio: 1.01 kali lipat
  Buang barisnya     meleset   0.2% dari rata-rata sebenarnya | std   770.5 (asli 771.9)
  Isi dengan mean    meleset   0.2% dari rata-rata sebenarnya | std   711.0 (asli 771.9)
  Isi dengan median  meleset  13.3% dari rata-rata sebenarnya | std   714.0 (asli 771.9)

=== MNAR (yang besar disembunyikan) ===
  new_cases yang TERLIHAT     — rata-rata:    175.8
  new_cases yang DISEMBUNYIKAN— rata-rata:    371.9
  Rasio: 2.12 kali lipat
  Buang barisnya     meleset  15.1% dari rata-rata sebenarnya | std   705.8 (asli 771.9)
  Isi dengan mean    meleset  15.1% dari rata-rata sebenarnya | std   647.0 (asli 771.9)
  Isi dengan median  meleset  27.4% dari rata-rata sebenarnya | std   649.7 (asli 771.9)



> **Baca angka-angka itu pelan-pelan.**
>
> **Bila hilangnya acak (MCAR):** nilai yang terlihat dan yang disembunyikan rata-ratanya nyaris sama
> (rasio sekitar 1,0). Membuang baris dan mengisi dengan mean hanya meleset sekitar **0,2%**. Tetapi
> mengisi dengan **median** meleset sekitar **13%**, karena `new_cases` sangat condong ke kanan: median (24)
> jauh di bawah mean (207). Mengisi dengan mean juga **mengecilkan sebaran**: simpangan baku turun dari
> sekitar 772 menjadi sekitar 711, padahal tidak ada nilai baru yang sungguh diketahui.
>
> **Bila yang besar lebih sering disembunyikan (MNAR):** nilai yang disembunyikan rata-ratanya sekitar
> **2,1 kali** nilai yang terlihat. Membuang baris dan mengisi dengan mean sama-sama meleset sekitar **15%**,
> dan median bahkan sekitar **27%**. Membuang baris tidak lebih aman daripada mengisi: keduanya menghapus
> bagian atas sebaran.
>
> Pada data MNAR, tidak ada teknik statistik yang bisa menyelamatkan Anda. Yang bisa dilakukan hanya dua:
> **kembali ke sumber data untuk melengkapi**, atau **melaporkan keterbatasannya secara terbuka** —
> misalnya menulis "rata-rata kasus harian minimal sekitar 176; angka sebenarnya lebih tinggi karena
> hari-hari dengan lonjakan besar cenderung tidak terlapor".
>
> Kalimat seperti itu, di laporan Anda, lebih bernilai daripada angka mana pun. (Angka tepatnya bisa sedikit
> berbeda bila Anda mengganti *seed*; polanya tidak.)

In [22]:
# CONTOH 7.3 — keputusan kita untuk dataset ini
# Kolom yang tidak berguna: dibuang, bukan diisi.
KOLOM_BUANG = {
    'city_or_regency': '100% kosong; tidak ada informasi sama sekali',
    'province'       : 'identik dengan location pada seluruh baris provinsi (redundan)',
    'country'        : 'konstan (Indonesia)',
    'continent'      : 'konstan (Asia)',
    'location_level' : 'konstan (Province) setelah baris nasional dipisahkan',
}
for kol, alasan in KOLOM_BUANG.items():
    if kol == 'province':
        assert (d['province'] == d['location']).all()      # buktikan redundansinya dulu, baru dibuang
    d = d.drop(columns=kol)
catat('Buang kolom kosong, konstan, atau redundan', len(KOLOM_BUANG),
      '5 kolom tidak membawa informasi (city_or_regency, province, country, continent, location_level); yang dihitung KOLOM')

# growth_factor_*: kosong karena pembagian dengan nol, BUKAN karena hilang. Dibiarkan kosong.
urut = d.sort_values(['location', 'date'])
for kol, sumber in [('growth_factor_of_new_cases', 'new_cases'), ('growth_factor_of_new_deaths', 'new_deaths')]:
    kemarin = urut.groupby('location')[sumber].shift().reindex(d.index)
    kosong  = d[kol].isna()
    dijelaskan = kosong & ((kemarin == 0) | kemarin.isna())
    print(f'{kol}: kosong {int(kosong.sum())}, dijelaskan hari pertama/kemarin nol {int(dijelaskan.sum())}')
    catat(f'{kol} dibiarkan kosong', kosong.sum(),
          'Kosong karena pembagi nol (kemarin 0 kasus) atau hari pertama; tidak ada nilai yang benar untuk diisi')

# special_status: kosong berarti provinsi tidak punya status khusus. Kekosongan itu sendiri sebuah kategori.
n_status = int(d['special_status'].isna().sum())
d['special_status'] = d['special_status'].fillna('Tidak ada status khusus')
d['punya_status_khusus'] = d['special_status'] != 'Tidak ada status khusus'
catat('special_status kosong menjadi kategori "Tidak ada status khusus"', n_status,
      'Kosong bukan hilang: provinsi itu memang tidak punya status khusus; juga dibuat penanda biner')

# total_cities / total_urban_villages / total_rural_villages: kosong karena tidak berlaku.
for kol, wilayah in [('total_cities', 'Sulawesi Barat'), ('total_urban_villages', 'Aceh'), ('total_rural_villages', 'DKI Jakarta')]:
    print(f'{kol}: kosong di', d.loc[d[kol].isna(), 'location'].unique().tolist())
    catat(f'{kol} dibiarkan kosong', d[kol].isna().sum(),
          f'Kosong hanya pada satu provinsi ({wilayah}) dan konstan sepanjang waktu: struktural, bukan acak. Tidak diisi 0 karena penyebabnya belum dipastikan')

  Buang kolom kosong, konstan, atau redundan: 5
growth_factor_of_new_cases: kosong 1935, dijelaskan hari pertama/kemarin nol 1935
  growth_factor_of_new_cases dibiarkan kosong: 1935
growth_factor_of_new_deaths: kosong 3442, dijelaskan hari pertama/kemarin nol 3442
  growth_factor_of_new_deaths dibiarkan kosong: 3442
  special_status kosong menjadi kategori "Tidak ada status khusus": 26335
total_cities: kosong di ['Sulawesi Barat']
  total_cities dibiarkan kosong: 901
total_urban_villages: kosong di ['Aceh']
  total_urban_villages dibiarkan kosong: 904
total_rural_villages: kosong di ['DKI Jakarta']
  total_rural_villages dibiarkan kosong: 929


### TUGAS 2 — hasil pembersihan

In [23]:
# TUGAS 2
d = d.reset_index(drop=True)

p5_baris_bersih = len(d)                                              # Jumlah baris setelah seluruh pembuangan/pemisahan
p5_dibuang      = len(mentah) - len(d)                                # Berapa baris yang hilang dibandingkan data mentah
p5_curiga       = (d['kumulatif_janggal'] == True).sum()              # Berapa baris bertanda kumulatif_janggal
p5_rate_kosong  = d['case_recovered_rate'].isna().sum()               # Berapa nilai kosong (NaN) pada case_recovered_rate setelah rasio > 100% ditangani

cek('P5_baris_bersih', p5_baris_bersih)
cek('P5_dibuang', p5_dibuang)
cek('P5_curiga', p5_curiga)
cek('P5_rate_kosong', p5_rate_kosong)

BENAR   P5_baris_bersih = 30893
BENAR   P5_dibuang = 929
BENAR   P5_curiga = 479
BENAR   P5_rate_kosong = 288


---
## Bagian 8 — Kolom turunan dan encoding

In [24]:
# CONTOH 8.1 — kolom turunan
# Urutkan dulu: rata-rata bergulir hanya bermakna bila baris tiap lokasi berurutan menurut tanggal.
d = d.sort_values(['location', 'date']).reset_index(drop=True)

d['bulan'] = d['date'].dt.to_period('M').astype(str)
d['tahun'] = d['date'].dt.year
d['hari']  = d['date'].dt.day_name()
d['kasus_rata7'] = (d.groupby('location')['new_cases']
                      .transform(lambda s: s.rolling(7, min_periods=7).mean()))

catat('Urutkan menurut lokasi dan tanggal, buat bulan/tahun/hari/kasus_rata7', len(d),
      'Rata-rata 7 hari meredam pola pelaporan mingguan; enam hari pertama tiap lokasi sengaja dibiarkan kosong, bukan diisi')

d[['date', 'location', 'bulan', 'hari', 'new_cases', 'kasus_rata7']].iloc[[0, 1, 6, 7]]

  Urutkan menurut lokasi dan tanggal, buat bulan/tahun/hari/kasus_rata7: 30893


,date,location,bulan,hari,new_cases,kasus_rata7
0,2020-03-26,Aceh,2020-03,Thursday,1,NaN
1,2020-03-27,Aceh,2020-03,Friday,3,NaN
6,2020-04-01,Aceh,2020-04,Wednesday,0,0.714286
7,2020-04-02,Aceh,2020-04,Thursday,0,0.571429


In [25]:
# CONTOH 8.2 — binning: angka menjadi kelompok
# Memakai kasus per juta penduduk agar provinsi besar dan kecil dapat dibandingkan.
d['kelas_kasus'] = pd.cut(d['new_cases_per_million'],
                          bins=[-0.01, 1, 10, 30, np.inf],
                          labels=['Sangat rendah', 'Rendah', 'Sedang', 'Tinggi'])
print(d['kelas_kasus'].value_counts().sort_index())

catat('Binning new_cases_per_million menjadi kelas_kasus', d['kelas_kasus'].notna().sum(),
      'Batas 1, 10, 30 per juta adalah keputusan, bukan perhitungan; dipilih agar tiap kelas cukup terisi')

kelas_kasus
Sangat rendah     8480
Rendah           10433
Sedang            5774
Tinggi            6206
Name: count, dtype: int64
  Binning new_cases_per_million menjadi kelas_kasus: 30893


> **Batas bin adalah keputusan, bukan perhitungan.** Angka 1, 10, dan 30 kasus per juta penduduk di atas dipilih
> dengan melihat sebaran (median sekitar 5,5, kuartil atas sekitar 22), bukan dari rumus. Tuliskan alasannya di
> *cleaning log*, karena mengubah batas bin akan mengubah kesimpulan analisis Anda.

In [26]:
# CONTOH 8.3 — encoding, dan kapan masing-masing dipakai
# Ordinal: ada urutan, jadi pakai kategori berurutan
urut_kelas = ['Sangat rendah', 'Rendah', 'Sedang', 'Tinggi']
d['kelas_kasus'] = pd.Categorical(d['kelas_kasus'], categories=urut_kelas, ordered=True)

# Nominal: tidak ada urutan, jadi one-hot bila nanti dibutuhkan
contoh_onehot = pd.get_dummies(d['island'], prefix='pulau', dummy_na=True)
print(contoh_onehot.columns.tolist())
print()
print('Untuk EDA, one-hot jarang diperlukan. Simpan untuk tahap pemodelan.')

['pulau_Jawa', 'pulau_Kalimantan', 'pulau_Maluku', 'pulau_Nusa Tenggara', 'pulau_Papua', 'pulau_Sulawesi', 'pulau_Sumatera', 'pulau_<NA>']

Untuk EDA, one-hot jarang diperlukan. Simpan untuk tahap pemodelan.


> **Jangan melakukan encoding hanya karena bisa.** Pada EDA, kategori lebih mudah dibaca daripada kolom
> nol-satu. One-hot baru berguna saat masuk pemodelan, yang bukan wilayah mata kuliah ini.

---
## Bagian 9 — Menyusun ulang bentuk tabel

**Tidy data** berarti: satu baris satu observasi, satu kolom satu variabel, satu tabel satu satuan amatan.

In [27]:
# CONTOH 9.1 — pivot: dari panjang ke lebar (untuk dibaca manusia)
lebar = (d.pivot_table(index='bulan', columns='island', values='new_cases', aggfunc='sum')
          .fillna(0).round(0).astype(int))
lebar

island,Jawa,Kalimantan,Maluku,Nusa Tenggara,Papua,Sulawesi,Sumatera
bulan,,,,,,,
2020-03,844,40,2,14,11,58,51
2020-04,6115,565,61,436,231,644,606
2020-05,10442,1367,313,739,597,1730,2003
2020-06,16280,3115,1093,1642,1113,4644,2850
2020-07,31245,4836,1164,2782,1540,7115,4921
2020-08,41565,6185,1084,2509,1111,5747,8758
2020-09,67913,8094,1163,4491,3852,6871,20380
2020-10,72347,8568,1171,3804,4731,6752,26841
2020-11,85426,9848,942,3350,2104,6957,21095


In [28]:
# CONTOH 9.2 — melt: kembali ke panjang (untuk dianalisis mesin)
panjang = lebar.reset_index().melt(id_vars='bulan', var_name='island', value_name='kasus_baru')
print(panjang.shape)
panjang.head(6)

(217, 3)


,bulan,island,kasus_baru
0,2020-03,Jawa,844
1,2020-04,Jawa,6115
2,2020-05,Jawa,10442
3,2020-06,Jawa,16280
4,2020-07,Jawa,31245
5,2020-08,Jawa,41565


> **Kapan memakai yang mana.** Bentuk **lebar** enak dibaca manusia dan cocok untuk tabel di laporan.
> Bentuk **panjang** adalah yang dibutuhkan hampir semua fungsi grafik dan agregasi. Simpan data Anda
> dalam bentuk panjang, lalu ubah ke lebar hanya pada saat menampilkan.

In [29]:
# CONTOH 9.3 — merge: menggabungkan tabel lain
# Tabel ringkasan per provinsi, lalu digabungkan kembali ke data harian.
profil_provinsi = (d.groupby('location')
                    .agg(hari_tercatat=('date', 'count'),
                         total_kasus_baru=('new_cases', 'sum'),
                         puncak_harian=('new_cases', 'max'))
                    .reset_index())

gabung = d.merge(profil_provinsi, on='location', how='left', validate='many_to_one')
print('Sebelum:', d.shape, '-> sesudah:', gabung.shape)
profil_provinsi.head(3)

Sebelum: (30893, 40) -> sesudah: (30893, 43)


,location,hari_tercatat,total_kasus_baru,puncak_harian
0,Aceh,904,44038,459
1,Bali,919,166831,2556
2,Banten,924,333875,7283


> **`validate='many_to_one'` itu rem darurat.** Tanpa itu, kunci yang ternyata tidak unik akan membuat
> baris berlipat ganda tanpa peringatan, dan total kasus Anda membengkak. Selalu nyatakan hubungan yang
> Anda harapkan; biarkan pandas yang menolak bila ternyata keliru.
>
> Periksa juga jumlah baris sebelum dan sesudah `merge`. Kalau bertambah, ada yang salah.

---
## Bagian 10 — Pipeline yang dapat dijalankan ulang

Sel-sel yang berserakan tidak bisa dijalankan ulang dengan hasil yang sama. Satukan menjadi satu fungsi.

In [30]:
# CONTOH 10.1 — pipeline: seluruh langkah Bagian 3-8 dalam satu fungsi
def bersihkan(df_mentah, path_nasional=None):
    """Terima data mentah, kembalikan data bersih (tingkat provinsi) beserta jejak keputusannya.
    Bila path_nasional diisi, baris tingkat nasional disimpan ke berkas itu."""
    x = df_mentah.copy()
    log = []
    def c(langkah, n, alasan):
        log.append({'langkah': langkah, 'baris_terdampak': int(n), 'alasan': alasan})

    # 1 tipe data
    x.columns = [snake(k) for k in x.columns]
    c('Seragamkan nama kolom ke snake_case', len(x.columns),
      'Nama berspasi, berkurung, dan berkapital (mis. "Area (km2)") menyulitkan kode; yang dihitung di sini KOLOM, bukan baris')
    for kol in ['case_fatality_rate', 'case_recovered_rate']:
        berformat = x[kol].astype(str).str.endswith('%')
        x[kol] = pd.to_numeric(x[kol].astype(str).str.replace('%', '', regex=False).str.strip(), errors='coerce')
        c(f'Konversi {kol} ke angka', berformat.sum(),
          'Buang "%" agar kolom dapat dihitung; satuan tetap persen (0-100), bukan pecahan')
    x['date'] = pd.to_datetime(x['date'].astype(str), format='%m/%d/%Y', errors='coerce')
    c('Konversi date dengan format %m/%d/%Y', len(x),
      'dayfirst=True akan menukar bulan dan hari (3/1/2020 menjadi 3 Januari) tanpa memunculkan galat')

    # 2 spasi
    teks = [k for k in x.columns if x[k].dtype == object or pd.api.types.is_string_dtype(x[k])]
    n_sp = 0
    for kol in teks:
        s = x[kol].astype('string')
        n_sp += int((s != s.str.strip()).sum())
        x[kol] = s.str.strip()
    c('Buang spasi awal dan akhir kolom teks', n_sp, 'Spasi tersembunyi memecah groupby dan melambungkan nunique')

    # 3 placeholder
    n_ph = 0
    for kol in teks:
        s = x[kol].astype('string')
        m = s.str.lower().isin(CURIGA).fillna(False)
        n_ph += int(m.sum())
        if m.any():
            x.loc[m, kol] = pd.NA
    c('Placeholder teks menjadi kosong', n_ph, 'N/A, -, null, dan teks kosong bukan nilai sungguhan')
    n_se = 0
    for kol in HITUNGAN:
        m = x[kol].isin(SENTINEL)
        n_se += int(m.sum())
        if m.any():
            x.loc[m, kol] = np.nan
    c('Kode sentinel -999/-9999 pada kolom hitungan menjadi kosong', n_se,
      'Kode semacam ini merusak rata-rata tanpa pernah terdeteksi isna(); diperiksa dan tidak ditemukan')

    # 4 kategori
    sebelum_k = x['location'].nunique()
    x['location'] = x['location'].replace(PETA_KATEGORI)
    c('Seragamkan kategori', sebelum_k - x['location'].nunique(),
      'Tidak ada sinonim/salah ketik pada nama lokasi; peta kosong adalah temuan, bukan kelalaian. Huruf tidak diubah otomatis')

    # 5 duplikat
    n_p = int(x.duplicated().sum())
    x = x.drop_duplicates()
    c('Hapus duplikat persis', n_p, 'Seluruh kolom identik; satu baris disimpan (hasil pemeriksaan: tidak ada)')
    n_k = int(x.duplicated(subset=['date', 'location'], keep=False).sum())
    x = (x.sort_values('total_cases', ascending=False)
           .drop_duplicates(subset=['date', 'location'], keep='first').sort_index())
    c('Satukan duplikat menurut (date, location)', n_k,
      'Satu lokasi hanya boleh satu baris per hari; dipilih baris dengan total_cases terbesar (hasil pemeriksaan: tidak ada)')
    n_ko = int(x[['date', 'location']].isna().any(axis=1).sum())
    x = x[x[['date', 'location']].notna().all(axis=1)]
    c('Buang baris tanpa date atau location', n_ko,
      'Kunci utama kosong tidak dapat ditelusuri kembali ke sumber (hasil pemeriksaan: tidak ada)')
    rg = x.groupby('location')['date'].agg(['min', 'max', 'count'])
    n_bo = int((((rg['max'] - rg['min']).dt.days + 1) - rg['count']).sum())
    c('Periksa hari yang hilang di tengah deret tiap lokasi', n_bo,
      'Tidak ada hari bolong. Lokasi baru muncul saat kasus pertamanya, itu BUKAN data hilang melainkan belum ada, jadi tidak diisi nol')
    nas = x[x['location_level'] == 'Country'].copy()
    x = x[x['location_level'] == 'Province'].copy()
    c('Pisahkan baris tingkat nasional ke tabel sendiri', len(nas),
      'Baris Country adalah agregat provinsi; bila dijumlahkan bersama baris provinsi, setiap kasus terhitung dua kali. Disimpan terpisah, tidak dihapus')
    if path_nasional:
        os.makedirs(os.path.dirname(path_nasional) or '.', exist_ok=True)
        nas.to_csv(path_nasional, index=False)

    # 6 nilai mustahil
    for kol in ['case_recovered_rate', 'case_fatality_rate']:
        bk = x[kol] > 100
        n_r = int(bk.sum())
        x.loc[bk, kol] = np.nan
        c(f'{kol} > 100% menjadi kosong', n_r,
          'Rasio mustahil; berasal dari total sembuh/meninggal yang melebihi total kasus. Barisnya masih berguna untuk kolom lain')
    n_ng = 0
    for kol in HITUNGAN:
        bk = x[kol] < 0
        n_ng += int(bk.sum())
        if bk.any():
            x.loc[bk, kol] = np.nan
    c('Hitungan harian/kumulatif negatif menjadi kosong', n_ng,
      'Jumlah orang tidak bisa negatif; hasil pemeriksaan: tidak ada. Selisih negatif pada new_active_cases sengaja tidak disentuh')
    n_dm = int(((x['population'] <= 0) | (x['area_km2'] <= 0)).sum())
    c('Periksa populasi dan luas wilayah <= 0', n_dm, 'Penyebut untuk kolom per juta penduduk; hasil pemeriksaan: tidak ada')
    n_sl = int((x['total_active_cases'] - (x['total_cases'] - x['total_deaths'] - x['total_recovered'])).abs().gt(0).sum())
    c('Uji identitas total_active = cases - deaths - recovered', n_sl,
      'Identitas selalu terpenuhi, jadi aktif negatif adalah AKIBAT sembuh > kasus, bukan kesalahan hitung terpisah')
    x['kumulatif_janggal'] = ((x['total_recovered'] > x['total_cases']) |
                              (x['total_deaths'] > x['total_cases']) |
                              (x['total_active_cases'] < 0))
    c('Tandai baris kumulatif janggal', x['kumulatif_janggal'].sum(),
      'Sembuh/meninggal melebihi total kasus atau aktif negatif; ditandai, tidak ditimpa, karena belum diketahui kolom mana yang keliru')
    ur = x.sort_values(['location', 'date'])
    n_tk = 0
    for tot, baru in [('total_cases', 'new_cases'), ('total_deaths', 'new_deaths'), ('total_recovered', 'new_recovered')]:
        sl = ur.groupby('location')[tot].diff().reindex(x.index)
        n_tk += int((sl < 0).sum())
        n_tk += int((sl.notna() & (sl != x[baru])).sum())
    c('Uji konsistensi kolom kumulatif dan harian', n_tk,
      'Total tidak pernah turun dan selisihnya sama dengan new_*; hasil pemeriksaan: konsisten. Anomali ada di rasio sembuh/kasus, bukan di hitungan harian')

    # 7 nilai kosong
    x = x.drop(columns=list(KOLOM_BUANG))
    c('Buang kolom kosong, konstan, atau redundan', len(KOLOM_BUANG),
      '5 kolom tidak membawa informasi (city_or_regency, province, country, continent, location_level); yang dihitung KOLOM')
    for kol in ['growth_factor_of_new_cases', 'growth_factor_of_new_deaths']:
        c(f'{kol} dibiarkan kosong', x[kol].isna().sum(),
          'Kosong karena pembagi nol (kemarin 0 kasus) atau hari pertama; tidak ada nilai yang benar untuk diisi')
    n_st = int(x['special_status'].isna().sum())
    x['special_status'] = x['special_status'].fillna('Tidak ada status khusus')
    x['punya_status_khusus'] = x['special_status'] != 'Tidak ada status khusus'
    c('special_status kosong menjadi kategori "Tidak ada status khusus"', n_st,
      'Kosong bukan hilang: provinsi itu memang tidak punya status khusus; juga dibuat penanda biner')
    for kol, wil in [('total_cities', 'Sulawesi Barat'), ('total_urban_villages', 'Aceh'), ('total_rural_villages', 'DKI Jakarta')]:
        c(f'{kol} dibiarkan kosong', x[kol].isna().sum(),
          f'Kosong hanya pada satu provinsi ({wil}) dan konstan sepanjang waktu: struktural, bukan acak. Tidak diisi 0 karena penyebabnya belum dipastikan')

    # 8 kolom turunan
    x = x.reset_index(drop=True).sort_values(['location', 'date']).reset_index(drop=True)
    x['bulan'] = x['date'].dt.to_period('M').astype(str)
    x['tahun'] = x['date'].dt.year
    x['hari'] = x['date'].dt.day_name()
    x['kasus_rata7'] = x.groupby('location')['new_cases'].transform(lambda s: s.rolling(7, min_periods=7).mean())
    c('Urutkan menurut lokasi dan tanggal, buat bulan/tahun/hari/kasus_rata7', len(x),
      'Rata-rata 7 hari meredam pola pelaporan mingguan; enam hari pertama tiap lokasi sengaja dibiarkan kosong, bukan diisi')
    x['kelas_kasus'] = pd.cut(x['new_cases_per_million'], bins=[-0.01, 1, 10, 30, np.inf],
                              labels=['Sangat rendah', 'Rendah', 'Sedang', 'Tinggi'])
    c('Binning new_cases_per_million menjadi kelas_kasus', x['kelas_kasus'].notna().sum(),
      'Batas 1, 10, 30 per juta adalah keputusan, bukan perhitungan; dipilih agar tiap kelas cukup terisi')
    x['kelas_kasus'] = pd.Categorical(x['kelas_kasus'], categories=['Sangat rendah', 'Rendah', 'Sedang', 'Tinggi'], ordered=True)

    return x.reset_index(drop=True), pd.DataFrame(log)

print('Fungsi bersihkan() siap. Salin langkah-langkah di atas ke dalam fungsi ini untuk notebook M2 Anda.')

Fungsi bersihkan() siap. Salin langkah-langkah di atas ke dalam fungsi ini untuk notebook M2 Anda.


> **Uji reproduktibilitas yang murah:** jalankan `bersihkan(mentah)` dua kali, lalu bandingkan hasilnya.
> Kalau berbeda, ada yang bergantung pada urutan acak atau pada keadaan sel sebelumnya. Pada pertemuan 15
> tim lain akan menjalankan ulang notebook Anda, jadi masalah ini lebih murah diselesaikan sekarang.

In [31]:
# CONTOH 10.1b — uji reproduktibilitas
hasil_1, log_1 = bersihkan(mentah)
hasil_2, log_2 = bersihkan(mentah)

print('Dua kali jalan menghasilkan data identik :', hasil_1.equals(hasil_2))
print('Dua kali jalan menghasilkan log identik  :', log_1.equals(log_2))
print('Sama dengan hasil langkah-demi-langkah    :', hasil_1.equals(d), '| jumlah langkah log sama:', len(log_1) == len(jejak))
print('Data mentah tidak berubah                 :', mentah.shape, '| kolom pertama masih:', repr(mentah.columns[0]))

Dua kali jalan menghasilkan data identik : True
Dua kali jalan menghasilkan log identik  : True
Sama dengan hasil langkah-demi-langkah    : True | jumlah langkah log sama: True
Data mentah tidak berubah                 : (31822, 38) | kolom pertama masih: 'Date'


In [32]:
# CONTOH 10.2 — simpan hasil
os.makedirs('data/processed', exist_ok=True)
os.makedirs('reports', exist_ok=True)
os.makedirs('notebooks', exist_ok=True)

d.to_csv('data/processed/covid_provinsi_bersih.csv', index=False)
nasional.to_csv('data/processed/covid_nasional.csv', index=False)
print('Tersimpan (provinsi):', d.shape)
print('Tersimpan (nasional):', nasional.shape)
print()
print('Data mentah tidak disentuh:', mentah.shape, '->', RAW)

Tersimpan (provinsi): (30893, 40)
Tersimpan (nasional): (929, 38)

Data mentah tidak disentuh: (31822, 38) -> data/raw/covid_19_indonesia_time_series_all.csv


---
## Bagian 11 — Cleaning log

Dokumen ini yang membedakan pembersihan yang bisa dipertanggungjawabkan dari pembersihan yang asal jalan.
Setiap baris menjawab tiga hal: **apa yang diubah, berapa banyak, dan mengapa**.

In [33]:
# CONTOH 11.1 — cetak log
log = pd.DataFrame(jejak)
log.index = range(1, len(log) + 1)
log

,langkah,baris_terdampak,alasan
1,Seragamkan nama kolom ke snake_case,38,"Nama berspasi, berkurung, dan berkapital (mis...."
2,Konversi case_fatality_rate ke angka,31822,"Buang ""%"" agar kolom dapat dihitung; satuan te..."
3,Konversi case_recovered_rate ke angka,31822,"Buang ""%"" agar kolom dapat dihitung; satuan te..."
4,Konversi date dengan format %m/%d/%Y,31822,dayfirst=True akan menukar bulan dan hari (3/1...
5,Buang spasi awal dan akhir kolom teks,0,Spasi tersembunyi memecah groupby dan melambun...
6,Placeholder teks menjadi kosong,0,"N/A, -, null, dan teks kosong bukan nilai sung..."
7,Kode sentinel -999/-9999 pada kolom hitungan m...,0,Kode semacam ini merusak rata-rata tanpa perna...
8,Seragamkan kategori,0,Tidak ada sinonim/salah ketik pada nama lokasi...
9,Hapus duplikat persis,0,Seluruh kolom identik; satu baris disimpan (ha...
10,"Satukan duplikat menurut (date, location)",0,Satu lokasi hanya boleh satu baris per hari; d...


In [34]:
# CONTOH 11.2 — simpan sebagai Markdown untuk repositori
baris = ['# Cleaning Log', '',
         'Dataset: covid_19_indonesia_time_series_all.csv  ',
         f'Baris mentah: {len(mentah)} -> baris bersih (provinsi): {len(d)}; baris nasional dipisahkan: {len(nasional)}  ',
         '', '| # | Langkah | Baris terdampak | Alasan |', '|---|---|---|---|']
for i, r in log.iterrows():
    baris.append(f"| {i} | {r['langkah']} | {r['baris_terdampak']} | {r['alasan']} |")

with open('cleaning_log.md', 'w', encoding='utf-8') as f:
    f.write('\n'.join(baris))

print('\n'.join(baris[:10]))

# Cleaning Log

Dataset: covid_19_indonesia_time_series_all.csv  
Baris mentah: 31822 -> baris bersih (provinsi): 30893; baris nasional dipisahkan: 929  

| # | Langkah | Baris terdampak | Alasan |
|---|---|---|---|
| 1 | Seragamkan nama kolom ke snake_case | 38 | Nama berspasi, berkurung, dan berkapital (mis. "Area (km2)") menyulitkan kode; yang dihitung di sini KOLOM, bukan baris |
| 2 | Konversi case_fatality_rate ke angka | 31822 | Buang "%" agar kolom dapat dihitung; satuan tetap persen (0-100), bukan pecahan |
| 3 | Konversi case_recovered_rate ke angka | 31822 | Buang "%" agar kolom dapat dihitung; satuan tetap persen (0-100), bukan pecahan |


### TUGAS 3 — kelengkapan log

In [35]:
# TUGAS 3
p5_langkah  = len(log)                           # berapa langkah yang tercatat di cleaning log Anda
p5_rate_num = d['case_fatality_rate'].notna().sum()   # berapa baris yang case_fatality_rate-nya sudah berupa angka
p5_bulan    = d['bulan'].nunique()               # berapa bulan berbeda yang muncul

cek('P5_langkah', p5_langkah)
cek('P5_rate_num', p5_rate_num)
cek('P5_bulan', p5_bulan)

BENAR   P5_langkah = 29
BENAR   P5_rate_num = 30850
BENAR   P5_bulan = 31


> Kalau jumlah langkah Anda berbeda dari kunci, itu **tidak otomatis salah** — tim boleh memecah atau
> menggabungkan langkah. Yang penting setiap perubahan ada catatannya. Angka pada kunci mengikuti
> pembagian langkah di workbook ini.

---
## Bagian 12 — Verifikasi: jalankan ulang diagnosis pertemuan 4

Pembersihan belum selesai sampai Anda membuktikan bahwa masalahnya benar-benar hilang —
dan bahwa tidak ada masalah baru yang Anda ciptakan sendiri.

In [36]:
# CONTOH 12.1 — diagnosis ulang
def angka(seri):
    return pd.to_numeric(seri.astype(str).str.rstrip('%'), errors='coerce')

def diagnosa_singkat(data, nama):
    x = data.rename(columns=snake)
    print(f'=== {nama} ===')
    print('  Baris                          :', len(x))
    print('  Duplikat persis                :', int(x.duplicated().sum()))
    print('  Lokasi unik                    :', x['location'].astype(str).nunique())
    print('  Baris bernama "Indonesia"      :', int((x['location'] == 'Indonesia').sum()), '(agregat nasional)')
    print('  date bertipe datetime          :', pd.api.types.is_datetime64_any_dtype(x['date']))
    print('  rasio sembuh bertipe angka     :', pd.api.types.is_numeric_dtype(x['case_recovered_rate']))
    print('  rasio sembuh > 100%            :', int((angka(x['case_recovered_rate']) > 100).sum()))
    print('  rasio kematian > 100%          :', int((angka(x['case_fatality_rate']) > 100).sum()))
    print('  total_active_cases < 0         :', int((x['total_active_cases'] < 0).sum()))
    print()

diagnosa_singkat(mentah, 'SEBELUM')
diagnosa_singkat(d, 'SESUDAH')

=== SEBELUM ===
  Baris                          : 31822
  Duplikat persis                : 0
  Lokasi unik                    : 35
  Baris bernama "Indonesia"      : 929 (agregat nasional)
  date bertipe datetime          : False
  rasio sembuh bertipe angka     : False
  rasio sembuh > 100%            : 288
  rasio kematian > 100%          : 43
  total_active_cases < 0         : 479

=== SESUDAH ===
  Baris                          : 30893
  Duplikat persis                : 0
  Lokasi unik                    : 34
  Baris bernama "Indonesia"      : 0 (agregat nasional)
  date bertipe datetime          : True
  rasio sembuh bertipe angka     : True
  rasio sembuh > 100%            : 0
  rasio kematian > 100%          : 0
  total_active_cases < 0         : 479



In [37]:
# CONTOH 12.2 — apa yang HILANG dan apa yang BERTAMBAH
m = mentah.rename(columns=snake)
kolom_sama = [k for k in m.columns if k in d.columns]

banding = pd.DataFrame({
    'kosong_sebelum': m[kolom_sama].isna().sum(),
    'kosong_sesudah': d[kolom_sama].isna().sum(),
})
banding['persen_sebelum'] = (banding['kosong_sebelum'] / len(m) * 100).round(2)
banding['persen_sesudah'] = (banding['kosong_sesudah'] / len(d) * 100).round(2)
banding['selisih_persen'] = (banding['persen_sesudah'] - banding['persen_sebelum']).round(2)

print('Kolom yang dibuang :', [k for k in m.columns if k not in d.columns])
print('Kolom yang ditambah:', [k for k in d.columns if k not in m.columns])
print()
banding[banding['selisih_persen'] != 0].sort_values('selisih_persen', ascending=False)

Kolom yang dibuang : ['location_level', 'city_or_regency', 'province', 'country', 'continent']
Kolom yang ditambah: ['kumulatif_janggal', 'punya_status_khusus', 'bulan', 'tahun', 'hari', 'kasus_rata7', 'kelas_kasus']



,kosong_sebelum,kosong_sesudah,persen_sebelum,persen_sesudah,selisih_persen
case_recovered_rate,0,288,0.00,0.93,0.93
growth_factor_of_new_deaths,3447,3442,10.83,11.14,0.31
growth_factor_of_new_cases,1939,1935,6.09,6.26,0.17
case_fatality_rate,0,43,0.00,0.14,0.14
total_urban_villages,904,904,2.84,2.93,0.09
total_rural_villages,929,929,2.92,3.01,0.09
total_cities,901,901,2.83,2.92,0.09
island,929,0,2.92,0.00,-2.92
time_zone,929,0,2.92,0.00,-2.92
special_status,27264,0,85.68,0.00,-85.68


> **Baca tabel itu dengan teliti, karena arahnya tidak seragam.**
>
> - **Bertambah, dan itu wajar:** `case_recovered_rate` naik dari 0 menjadi 288 baris kosong dan
>   `case_fatality_rate` dari 0 menjadi 43. Rasio mustahil yang tadinya menyamar sebagai angka kini jujur mengaku
>   kosong. Data yang lebih jujur memang terlihat lebih buruk, dan itulah tandanya pembersihan berhasil.
> - **Turun, tetapi karena alasan berbeda:** `island` dan `time_zone` kosong hanya pada 929 baris nasional yang
>   sudah dipisahkan. `special_status` turun dari 85,7% ke 0% karena kekosongannya **diberi arti** sebagai kategori
>   "Tidak ada status khusus", bukan karena datanya ditemukan.
> - **Naik sedikit padahal tidak ada yang berubah:** `growth_factor_*` dan `total_cities` dan sebagainya hanya
>   bergeser karena **penyebutnya** (jumlah baris) mengecil setelah baris nasional dipisahkan. Selisih persen
>   kecil pada kolom yang jumlah kosongnya sama bukan masalah baru.
> - **Tidak berubah, dan itu disengaja:** `total_active_cases < 0` masih 479 baris. Nilainya hanya **ditandai**
>   (`kumulatif_janggal`), tidak ditimpa.
>
> Laporkan angka ini apa adanya di M2. Laporan yang mengaku "rasio sembuh kosong bertambah dari 0 menjadi 288
> baris karena rasio mustahil diungkap" jauh lebih meyakinkan daripada laporan yang semuanya nol.

---
## Bagian 13 — Kerjakan untuk proyek tim: M2

**M2 — Data Quality Report, bobot 10%.** Tiga berkas yang dikumpulkan:

| Berkas | Isi | Diperiksa oleh rubrik |
|---|---|---|
| `reports/data_quality_report.docx` | Temuan, keputusan, keterbatasan | Kelengkapan diagnosis (25), Justifikasi (30) |
| `notebooks/02_cleaning.ipynb` | Pipeline yang dapat dijalankan ulang | Ketepatan implementasi (20) |
| `cleaning_log.md` | Setiap langkah, jumlah, dan alasan | Dokumentasi dan keterlacakan (15) |

Dimensi kelima, **Penanganan data pribadi (10)**, dinilai dari apakah data olahan Anda tetap aman —
yaitu lanjutan pekerjaan pertemuan 3. *Catatan untuk dataset ini:* data COVID-19 ini sudah teragregasi per
provinsi per hari, tidak memuat nama atau identitas individu, tetapi tulis pemeriksaan itu secara eksplisit di laporan.

Gunakan `TEMPLATE_Data_Quality_Report.docx` sebagai kerangka laporan.

In [38]:
# 13.1 — salin pola workbook ini ke dataset tim
df_tim = pd.read_csv('data/raw/covid_19_indonesia_time_series_all.csv', encoding='utf-8-sig')
bersih_tim, log_tim = bersihkan(df_tim, path_nasional='data/processed/covid_nasional.csv')
bersih_tim.to_csv('data/processed/covid_provinsi_bersih.csv', index=False)
print('Selesai:', bersih_tim.shape, '| langkah log:', len(log_tim))

Selesai: (30893, 40) | langkah log: 29


In [39]:
# 13.2 — jalankan pemeriksa kelengkapan sebelum mengumpulkan
# Dari terminal, di dalam folder repositori tim:
#     python cek_m2.py
# Skrip akan memeriksa keberadaan berkas, struktur folder, dan isi minimal cleaning log.

### Daftar periksa M2

- [ ] Seluruh temuan pertemuan 4 muncul di laporan, termasuk yang Anda putuskan untuk dibiarkan
- [ ] Setiap keputusan punya alasan tertulis, bukan sekadar deskripsi tindakan
- [ ] Minimal satu keputusan menyebutkan **alternatif yang dipertimbangkan dan ditolak**
- [ ] Dampak terhadap sebaran data ditunjukkan untuk minimal satu kolom (pola Bagian 7)
- [ ] Keterbatasan dinyatakan terbuka, termasuk kolom yang tidak bisa diselamatkan
- [ ] `data/raw/` tidak berubah sama sekali dan tidak ikut diunggah
- [ ] Notebook berjalan dari awal sampai akhir tanpa galat setelah *Restart and run all*
- [ ] `cleaning_log.md` ada di akar repositori
- [ ] `python cek_m2.py` berjalan tanpa keluhan

> **Kesalahan yang paling mahal pada M2:** menulis apa yang dilakukan, bukan mengapa.
> "Kolom X diisi dengan median" adalah deskripsi. "Kolom X diisi dengan median karena kekosongannya
> tersebar merata di semua kelompok dan hanya 3%, sehingga pilihan ini tidak menggeser sebaran lebih dari
> 1%; alternatif membuang baris ditolak karena akan menghapus 48 transaksi yang kolom lainnya lengkap"
> adalah justifikasi. Rubrik memberi bobot 30 untuk yang kedua.

---
## Kalau Anda mentok

| Urutan | Yang dilakukan |
|---|---|
| 1 | Baca baris terakhir pesan galat |
| 2 | *Runtime → Restart and run all*. Banyak galat muncul karena sel dijalankan tidak berurutan. |
| 3 | Cetak `d.dtypes` dan `d.shape` sebelum serta sesudah langkah yang bermasalah |
| 4 | Tanyakan ke anggota tim |
| 5 | Tanyakan ke asisten saat sesi berlangsung |
| 6 | Tulis di forum tanya-jawab LMS; dosen menjawab setiap hari kerja |

**Galat yang paling sering muncul di pertemuan ini:**

| Pesan | Artinya | Perbaikannya |
|---|---|---|
| `can only use .str accessor with string values` | Kolomnya bukan teks | `.astype(str)` atau `.astype('string')` lebih dulu |
| `SettingWithCopyWarning` | Menulis ke salinan, bukan ke tabel aslinya | Pakai `.loc[baris, kolom] = nilai`, dan `.copy()` setelah menyaring |
| `could not convert string to float` | Masih ada teks di kolom angka | Pakai `errors='coerce'`, lalu hitung berapa yang gagal |
| `MergeError: Merge keys are not unique` | Kunci gabungan tidak unik | Itu justru gunanya `validate=`; perbaiki kuncinya, jangan hapus validasinya |
| `time data ... doesn't match format` | Format tanggal tidak sama dengan yang Anda tulis | Cetak beberapa nilai mentah `date`, cek urutan bulan/hari, sesuaikan `format=` |

---
## Ringkasan pertemuan 5

1. **Pembersihan adalah rangkaian keputusan.** Yang dinilai adalah alasannya, bukan pilihannya.
2. **Urutan menentukan hasil.** Tipe data dulu, nilai kosong paling akhir.
3. **Menandai lebih aman daripada menebak.** Kolom penanda memindahkan keputusan ke tahap yang tepat.
4. **Pada data MNAR, semua strategi meleset** — termasuk membuang baris. Yang menolong hanya kembali ke
   sumber data, atau melaporkan keterbatasannya secara terbuka.
5. **Nilai kosong yang bertambah bisa menjadi tanda keberhasilan**, bukan kegagalan: placeholder yang menyamar
   sebagai angka kini jujur mengaku kosong. Pada data COVID ini, yang bertambah adalah rasio mustahil yang diungkap.
6. **Pembersihan belum selesai sampai diagnosisnya dijalankan ulang.**

### Setelah pertemuan 5

- Selesaikan dan kumpulkan **M2** sesuai tenggat di LMS.
- Kerjakan kuis formatif pertemuan 5.
- Pertemuan 6 kembali bersama dosen: analisis univariat, *outlier*, dan visualisasi yang jujur.